In [1]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
import os

In [1]:
import json

In [2]:
def load_config(path: str = "train_config.json") -> dict:
    with open(path) as f:
        return json.load(f)


In [1]:
def format_example(example):
    """
    careful with trailing whitespaces.
    """
    if example["input"]:
        example['text'] = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{example["instruction"]}
                
### Input:
{example["input"]}

### Response:
{example["output"]}"""
    else:
        example['text'] =f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.  

### Instruction:
{example["instruction"]}

### Response:
{example["output"]}""" 
    example['text'] += tokenizer.eos_token ## training might fail if model don't have EOS token.
    return example

In [5]:
def load_training_dataset(dataset_url: str):
    print("Loading dataset …")
    dataset = load_dataset("json", data_files=dataset_url, split="train")

    dataset = dataset.map(format_example)
    
    print(f"Loaded {len(dataset)} examples")
    return dataset


In [ ]:
def set_quant_cfg(quant_cfg_dict: dict)
    if quant_cfg_dict:
        bits = quant_cfg_dict.get("bits", 4)
        if bits == 8:
            bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        elif bits == 4:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type=quant_cfg.get("quant_type", "nf4"),
                bnb_4bit_compute_dtype=dtype,
                bnb_4bit_use_double_quant=quant_cfg.get("double_quant", True),
            )
        else:
            raise ValueError(f"Unsupported bits: {bits}. Use 4 or 8.")


In [ ]:
def load_model(model_id: str, bnb_config = None, prepare_for_training: bool = True,
               use_bf16: bool = False):
    print("Loading model & tokenizer …")
    hf_token = os.environ.get("HF_TOKEN")
    dtype = torch.bfloat16 if use_bf16 else torch.float16

    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token,
        torch_dtype=dtype,
        attn_implementation="sdpa",
    )
    if prepare_for_training and bnb_config is not None:
        model = prepare_model_for_kbit_training(model)
    return model, tok

In [2]:
import os

In [3]:
import json

In [4]:
import requests

In [5]:
from peft import LoraConfig, PeftModel
# prepare_model_for_kbit_training -- not available in authors repo

In [6]:
from datasets import load_dataset

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [8]:
import lm_eval
from lm_eval.models.huggingface import HFLM

In [9]:
def load_config(path: str = "train_config.json") -> dict:
    with open(path) as f:
        return json.load(f)


In [10]:
def set_bnb_config(quant_cfg: dict = None, compute_dtype=torch.bfloat16):
    bnb_config = None
    if quant_cfg:
        bits = quant_cfg.get("bits", 4)
        if bits == 8:
            bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        elif bits == 4:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type=quant_cfg.get("quant_type", "nf4"),
                bnb_4bit_compute_dtype=compute_dtype,
                bnb_4bit_use_double_quant=quant_cfg.get("double_quant", True),
            )
        else:
            raise ValueError(f"Unsupported bits: {bits}. Use 4 or 8.")
    return bnb_config

In [11]:
def load_model(model_id: str, bnb_cfg = None, prepare_for_training: bool = True,
               use_bf16: bool = False):
    '''
    input: 
        bnb_cfg should be BitsAndBytesConfig object or None.
    '''
    
    print("Loading model & tokenizer …")
    hf_token = os.environ.get("HF_TOKEN")
    dtype = torch.bfloat16 if use_bf16 else torch.float16

    
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    if tokenizer.pad_token is None:
        print('setting pad_token to eos_token')
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right" # original authors also used right

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_cfg,
        device_map="auto",
        # device_map={"": 0},
        token=hf_token,
        torch_dtype=dtype,
        attn_implementation="sdpa"
    )
    if prepare_for_training and bnb_cfg is not None:
        raise NotImplementedError  # kbit_training not available in authors repo
    #     model = prepare_model_for_kbit_training(model)
    return model, tokenizer

In [12]:
## PARAMETERS
CONFIG = 'train/config_dora.json'
ADAPTER_PATH = '/teamspace/lightning_storage/models/llama3-8b-lora-commonsense-alpha64-v3/final_adapter/'
# ADAPTER_PATH = '/teamspace/studios/this_studio/DoRA-weights/llama_dora_commonsense_checkpoints/LLama3-8B/dora_r32/'
QUANT = None
MODEL_ID = 'meta-llama/Llama-3.2-1B'
# MODEL_ID = 'meta-llama/Meta-Llama-3-8B'
# MODEL_ID = 'Qwen/Qwen2.5-1.5B'

In [13]:
cfg = load_config(CONFIG)

In [14]:
bnb_config = set_bnb_config(None)
base, tokenizer = load_model(MODEL_ID, bnb_config, prepare_for_training=False, use_bf16=True)

Loading model & tokenizer …


`torch_dtype` is deprecated! Use `dtype` instead!


setting pad_token to eos_token


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [15]:
model = PeftModel.from_pretrained(base, ADAPTER_PATH)

In [16]:
def generate_prompt(instruction):
    return f"""Below is an instruction that describes a task. Write a response that appropriately completes the request. 

### Instruction:
{instruction}

### Response:
"""

### Original authors implementation

Requests JSON from LLM-Adapters repo, code used to join these examples.

In [17]:
import requests

In [18]:
import regex as re

In [19]:
device = next(model.parameters()).device

### Loop eval through the whole data

In [20]:
def extract_answer(bench: str, sentence: str) -> str:
    sentence_ = sentence.strip()

    if bench == 'boolq':
        pred_answers = re.findall(r'true|false', sentence_)

    elif bench == 'piqa':
        pred_answers = re.findall(r'solution1|solution2', sentence_)

    elif bench in ['social_i_qa', 'ARC-Challenge', 'ARC-Easy', 'openbookqa']:
        pred_answers = re.findall(r'answer1|answer2|answer3|answer4|answer5', sentence_)

    elif bench == 'hellaswag':
        pred_answers = re.findall(r'ending1|ending2|ending3|ending4', sentence_)

    elif bench == 'winogrande':
        pred_answers = re.findall(r'option1|option2', sentence_)

    else:
        return ""

    return pred_answers[0] if pred_answers else ""

In [21]:
# 1. Is the model fully on GPU?
print("Model device (first param):", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

# 2. Check for any CPU parameters (the smoking gun)
cpu_params = [(n, p.device) for n, p in model.named_parameters() if p.device.type == "cpu"]
print(f"CPU params: {len(cpu_params)} / {sum(1 for _ in model.parameters())}")
if cpu_params[:3]:
    print("Examples:", cpu_params[:3])

# 3. Check buffers too (LayerNorm stats, RoPE caches, etc.)
cpu_bufs = [(n, b.device) for n, b in model.named_buffers() if b.device.type == "cpu"]
print(f"CPU buffers: {len(cpu_bufs)}")
if cpu_bufs[:3]:
    print("Examples:", cpu_bufs[:3])

# 4. If loaded with accelerate / device_map="auto"
if hasattr(model, "hf_device_map"):
    print("hf_device_map:", model.hf_device_map)

# 5. Are inputs on the right device?
print("CUDA available:", torch.cuda.is_available())
print("Current device:", torch.cuda.current_device() if torch.cuda.is_available() else "n/a")
print("VRAM used (GB):", torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0)

Model device (first param): cuda:0
Model dtype: torch.bfloat16
CPU params: 0 / 611
CPU buffers: 0
CUDA available: True
Current device: 0
VRAM used (GB): 16.290948096


In [22]:
model = model.to("cuda")

In [23]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [24]:
import copy
import json


In [25]:
# url = "https://raw.githubusercontent.com/AGI-Edgerunners/LLM-Adapters/refs/heads/main/dataset/hellaswag/test.json"
# data = requests.get(url).json()

In [26]:
# benchmark = "Rowan/hellaswag"
benchmark = "google/boolq"
ds = load_dataset(benchmark, split="validation")

In [27]:


save_file = "eval_results_boolq.json"
bench = "boolq"
limit = None          # set e.g. 50 to smoke-test first
batch_size = 8       # tune: 8/16/32 depending on VRAM
num_beams = 4         # keep 4 to match paper; drop to 1 for ~4-8x speedup
max_new_tokens = 32

output_data = []
correct = 0
device = next(model.parameters()).device
total = limit if limit is not None else len(ds)

# ---- critical setup for batched causal-LM generation ----
# Left-pad so the last real token of every sequence sits at the right edge;
# otherwise generation continues from padding and you get garbage.
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()

# Materialize the subset we're actually iterating over
examples = []
for idx, data in enumerate(ds):
    if limit is not None and idx >= limit:
        break
    examples.append(data)

def build_instruction(data, bench):
    if bench == "hellaswag":
        activity = data["activity_label"]
        ctx = data["ctx"]
        endings = data["endings"]
        return (
            f"Please choose the correct ending to complete the given sentence: {activity}: {ctx}\n\n"
            f"Ending1: {endings[0]} Ending2: {endings[1]} Ending3: {endings[2]} Ending4: {endings[3]}\n\n"
            f"Answer format: ending1/ending2/ending3/ending4"
        )

    elif bench == "boolq":
        question = data["question"]
        return (
            f"Please answer the following question with true or false, "
            f"question: {question}\n"
            f"Answer format: true/false"
        )

In [28]:
processed = 0
for start in range(0, total, batch_size):
    batch = examples[start:start + batch_size]

    instructions = [build_instruction(d, bench) for d in batch]
    prompts = [generate_prompt(ins) for ins in instructions]
    if bench == "boolq":
        labels = [str(bool(d["answer"])).lower() for d in batch]
    elif bench == "hellaswag":
        labels = [f"ending{int(d['label']) + 1}" for d in batch]
    else:
        raise ValueError(f"Unsupported bench: {bench}")

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            num_beams=num_beams,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    gen_ids = output_ids[:, input_len:]
    output_texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

    for data, instruction, label, output_text in zip(batch, instructions, labels, output_texts):
        predict = extract_answer(bench, output_text)

        if bench == 'boolq':
            normalized_label = str(label).lower()
        else:
            normalized_label = label

        flag = normalized_label == predict
        if flag:
            correct += 1

        row = copy.deepcopy(dict(data))
        row["instruction"] = instruction
        row["answer"] = label
        row["output_pred"] = output_text
        row["pred"] = predict
        row["flag"] = flag
        output_data.append(row)

        processed += 1

    print(f"test:{processed}/{total} | correct {correct} | accuracy {correct / processed:.4f}")

print('---------------')
print(f"final accuracy: {correct}/{total} = {correct / total:.4f}")
print('---------------')
with open(save_file, 'w+') as f:
    json.dump(output_data, f, indent=4)

Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


test:8/3270 | correct 6 | accuracy 0.7500


Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


test:16/3270 | correct 11 | accuracy 0.6875


...

test:3248/3270 | correct 1887 | accuracy 0.5810


Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


test:3256/3270 | correct 1892 | accuracy 0.5811


Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


test:3264/3270 | correct 1894 | accuracy 0.5803
test:3270/3270 | correct 1899 | accuracy 0.5807
---------------
final accuracy: 1899/3270 = 0.5807
---------------


In [29]:
import json

In [30]:

with open("eval_results_boolq.json") as f:
    data = json.load(f)

correct = sum(1 for row in data if row["flag"])
total = len(data)
print(f"accuracy: {correct}/{total} = {correct/total:.4f}")


accuracy: 1899/3270 = 0.5807
